# KIT Summer School - Wave Phenomena: Analysis and Numerics

# Project 1: Vlasov-Poisson equation

Write the problem here

### Python importations

In [ ]:
# IMPORTATIONS
import numpy as np
import scipy.linalg as la
from matplotlib import pyplot as plt, cm, animation
from scipy.integrate import solve_ivp
from scipy import sparse as sparse
from tqdm import tqdm

# Landau damping

## Initial value

In [ ]:
def landau_damping(nx, nv, r):
    # SPACE
    L = 4*np.pi
    dx = L/nx
    xs = np.linspace(0, L-dx, nx)

    # VELOCITY
    dv = 12/nv
    vs = np.linspace(-6, 6-dv, nv)

    # INITIAL VALUE
    X = np.zeros((nx, r))
    S = np.zeros((r, r))
    V = np.zeros((nv, r))
    S[0,0] = 1
    X[:,0] = 1 + 1e-2*np.cos(0.5*xs)
    V[:,0] = np.exp(-0.5*vs**2)/np.sqrt(2*np.pi)

    return xs, dx, vs, dv, X, S, V

# PARAMETERS
r = 20
nx = 200
nv = 200

# GENERATE THE DATA
xs, dx, vs, dv, X, S, V = landau_damping(nx, nv, r)

# PLOT
fig = plt.imshow(X.dot(S.dot(V.T)))
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.show()

## Vector field of the three steps

Write the vector field here in LaTeX.

In [ ]:
# DIFFERENTIAL STENCIL
Ax = 1/(2*dx) * sparse.diags([1,-1,1,-1],[-nx+1,-1,1,nx-1], shape=(nx,nx))
Av = 1/(2*dv) * sparse.diags([1,-1,1,-1],[-nv+1,-1,1,nv-1], shape=(nv,nv))
diagV = np.diag(vs)

def K_ode(t, K, C1, diagE, C2):
    dK = - Ax.dot(K).dot(C1.T) + np.linalg.multi_dot([diagE, K, C2.T])
    return dK
def vec_K_ode(t, k, K_shape, C1, diagE, C2):
    K = np.reshape(k, K_shape)
    dK = K_ode(t,K, C1, diagE, C2)
    dk = dK.flatten()
    return dk

def S_ode(t, S, D2, C1, D1, C2):
    dS = np.linalg.multi_dot([D2, S, C1.T]) - np.linalg.multi_dot([D1, S, C2.T])
    return dS
def vec_S_ode(t, s, S_shape, D2, C1, D1, C2):
    S = np.reshape(s, S_shape)
    dS = S_ode(t, S, D2, C1, D1, C2)
    ds = dS.flatten()
    return ds

def L_ode(t, L, D1, D2):
    dL = Av.dot(L).dot(D1.T) - np.linalg.multi_dot([diagV, L, D2.T])
    return dL
def vec_L_ode(t, l, L_shape, D1, D2):
    L = np.reshape(l, L_shape)
    dL = L_ode(t, L, D1, D2)
    dl = dL.flatten()
    return dl


## DLRA routines

In [ ]:
def DLRA1_one_step(t_span, initial_value):
    # INITIALISATION
    h = t_span[1] - t_span[0]
    (X0,S0,V0) = initial_value

    # K STEP
    K0 = X0.dot(S0)
    K_shape = K0.shape
    # COMPUTE E
    dxE=(np.ones((nx,1))[:,0]).transpose()-dv*np.dot(K0,(V0.T.sum(axis=1)))
    # Electic field integration
    fftdxE= np.fft.fft(dxE)
    n =fftdxE.size
    freq = np.fft.fftfreq(n, d=dx)
    Einter=fftdxE/(1j*freq*2*np.pi)
    Einter[0]=0
    E=np.fft.ifft(Einter)
    diagE = np.diag(E)
    # COMPUTE C1, C2
    C1 = dv * V0.T.dot(np.diag(vs).dot(V0))
    C2 = dv * V0.T.dot(Av.dot(V0))
    # SOLVE THE K ODE
    sol_k = solve_ivp(vec_K_ode, (0,h), K0.flatten(), method='RK45', args=(K_shape, C1, diagE, C2))
    k1 = sol_k.y[:,-1]
    K1 = np.reshape(k1, K_shape)

    # QR of K
    X1, R1 = la.qr(K1, mode='economic')
    # RESCALE (because of the definition of the inner product)
    R1 = R1 * np.sqrt(dx)
    X1 = X1 / np.sqrt(dx)

    # S STEP
    S0 = R1
    S_shape = S0.shape
    # COMPUTE D1, D2
    D1 = X1.T.dot(diagE.dot(X1)) * dx
    D2 = X1.T.dot(Ax.dot(X1)) * dx
    # SOLVE THE S ODE
    sol_s = solve_ivp(vec_S_ode, (0,h), S0.flatten(), method='RK45', args=(S_shape, D2, C1, D1, C2))
    s1 = sol_s.y[:,-1]
    S1 = np.reshape(s1, S_shape)

    # L STEP
    L0 = V0.dot(S1.T)
    L_shape = L0.shape
    sol_l = solve_ivp(vec_L_ode, (0,h), L0.flatten(), method='RK45', args=(L_shape, D1, D2))
    l1 = sol_l.y[:,-1]
    L1 = np.reshape(l1, L_shape)

    # QR OF L
    Q2,R2 = la.qr(L1, mode='economic')
    # RESCALE (because of the definition of the inner product)
    R2 = R2 * np.sqrt(dv)
    Q2 = Q2 / np.sqrt(dv)

    # RETURN FINAL VALUE AND ELECTRIC ENERGY
    (X1, S1, V1) = (X1, R2.T, Q2)
    final_value = (X1, S1, V1)
    return final_value, E

def DLRA1_many_steps(t_span, initial_value, n_steps):
    ts = np.linspace(t_span[0], t_span[1], n_steps)
    all_values = [None for i in np.arange(n_steps)]
    E_values = np.zeros((n_steps,nx))
    all_values[0] = initial_value
    for i in tqdm(range(n_steps-1)):
        t_loc = (ts[i], ts[i+1])
        all_values[i+1], E_values[i] = DLRA1_one_step(t_loc, all_values[i])
    return all_values, E_values


## ELECTRIC ENERGY OF LANDAU DAMPING FOR SEVERAL RANKS

In [ ]:
# PARAMETERS
tstart = 0
tend = 40
time = (tstart, tend)
n_steps = 4000
ts = np.linspace(tstart, tend, n_steps)

# DLRA5
r1 = 2
xs, dx, vs, dv, X1, S1, V1 = landau_damping(nx, nv, r1)
initial_value1 = (X1, S1, V1)
_, E1 = DLRA1_many_steps(time, initial_value1, n_steps)

# DLRA10
r2 = 10
xs, dx, vs, dv, X2, S2, V2 = landau_damping(nx, nv, r2)
initial_value2 = (X2, S2, V2)
_, E2 = DLRA1_many_steps(time, initial_value2, n_steps)

# DLRA20
r3 = 20
xs, dx, vs, dv, X3, S3, V3 = landau_damping(nx, nv, r3)
initial_value3 = (X3, S3, V3)
_, E3 = DLRA1_many_steps(time, initial_value3, n_steps)


In [ ]:
# COMPUTE THE NORMS
norm_E1 = dx*la.norm(E1, axis=1)
norm_E2 = dx*la.norm(E2, axis=1)
norm_E3 = dx*la.norm(E3, axis=1)

# PLOT
fig = plt.figure(1, dpi=150)
plt.semilogy(ts, norm_E1, label=f'rank {r1}')
plt.semilogy(ts, norm_E2, label=f'rank {r2}')
plt.semilogy(ts, norm_E3, label=f'rank {r3}')
plt.title('Landau damping')
plt.xlabel('time')
plt.ylabel('electric energy')
plt.grid()
plt.legend()
plt.savefig('landau_damping_energy.pdf')
plt.show()

# TWO STREAM

In [ ]:
def two_stream(nx, nv, r):
    # SPACE
    L = 10*np.pi
    dx = L/nx
    xs = np.linspace(0, L-dx, nx)

    # VELOCITY
    dv = 12/nv
    vs = np.linspace(-6, 6-dv, nv)

    # INITIAL VALUE
    X = np.zeros((nx, r))
    S = np.zeros((r, r))
    V = np.zeros((nv, r))
    S[0,0] = 1
    X[:,0] = 1 + 1e-3*np.cos(0.2*xs)
    V[:,0] = 0.5*(np.exp(-0.5*(vs-2.4)**2) + np.exp(-0.5*(vs+2.4)**2)) /np.sqrt(2*np.pi)

    return xs, dx, vs, dv, X, S, V

# PARAMETERS
r = 20
nx = 200
nv = 200

# GENERATE THE DATA
xs, dx, vs, dv, X, S, V = two_stream(nx, nv, r)

# PLOT
fig = plt.imshow(X.dot(S.dot(V.T)))
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.show()

# APPLY DLRA WITH SEVERAL RANKS

In [ ]:
# PARAMETERS
tstart = 0
tend = 40
time = (tstart, tend)
n_steps = 4000
ts = np.linspace(tstart, tend, n_steps)

# DLRA5
r = 5
xs, dx, vs, dv, X5, S5, V5 = two_stream(nx, nv, r)
initial_value5 = (X5, S5, V5)
all_sol5, E5 = DLRA1_many_steps(time, initial_value5, n_steps)

# DLRA10
r = 10
xs, dx, vs, dv, X10, S10, V10 = two_stream(nx, nv, r)
initial_value10 = (X10, S10, V10)
all_sol10, E10 = DLRA1_many_steps(time, initial_value10, n_steps)

# DLRA20
r = 20
xs, dx, vs, dv, X20, S20, V20 = two_stream(nx, nv, r)
initial_value20 = (X20, S20, V20)
all_sol20, E20 = DLRA1_many_steps(time, initial_value20, n_steps)

## ELECTRIC ENERGY TWO STREAM

In [ ]:
# COMPUTE THE NORMS
norm_E5 = dx*la.norm(E5, axis=1)
norm_E10 = dx*la.norm(E10, axis=1)
norm_E20 = dx*la.norm(E20, axis=1)

# PLOT
fig = plt.figure(1, dpi=150)
plt.semilogy(ts, norm_E5, label='rank 5')
plt.semilogy(ts, norm_E10, label='rank 10')
plt.semilogy(ts, norm_E20, label='rank 20')
plt.title('Two stream')
plt.xlabel('time')
plt.ylabel('electric energy')
plt.grid()
plt.legend()
plt.savefig('two_stream_energy.pdf')
plt.show()

## PLOT OF THE SOLUTION

In [ ]:
final_sol = all_sol5[-1]
(X,S,V) = final_sol
F = X.dot(S.dot(V.T))

fig = plt.imshow(F)
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.title('rank 5')
plt.show()

In [ ]:
final_sol = all_sol10[-1]
(X,S,V) = final_sol
F = X.dot(S.dot(V.T))

fig = plt.imshow(F)
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.title('rank 10')
plt.show()

In [ ]:
final_sol = all_sol20[-1]
(X,S,V) = final_sol
F = X.dot(S.dot(V.T))


fig = plt.imshow(F)
fig.axes.get_xaxis().set_visible(False)
fig.axes.get_yaxis().set_visible(False)
plt.title('rank 20')
plt.show()

In [ ]:
from IPython.display import HTML
def animation_2D(sol_to_plot, title, do_save=False):
    nb_t_steps = int(sol_to_plot.shape[0]/20)

    Z = sol_to_plot

    # Function of updating
    def update_surf(frame_number, Z, plot):
        plot[0].remove()
        plot[0] = plt.imshow(Z[int(frame_number*20), :, :])
        plot[0].axes.get_xaxis().set_visible(False)
        plot[0].axes.get_yaxis().set_visible(False)

    # Figure and first image
    fig = plt.figure(dpi=150)
    plot = [plt.imshow(Z[0, :, :])]
    plot[0].axes.get_xaxis().set_visible(False)
    plot[0].axes.get_yaxis().set_visible(False)
    anim2d = animation.FuncAnimation(fig, update_surf, nb_t_steps, fargs=(Z, plot))
    if do_save:
        anim2d.save(title, fps=30)
    return anim2d

## RANK 5 ANIMATION

In [ ]:
# ASSEMBLE SOLUTION
sol_to_plot = np.zeros((n_steps, nx, nv))
for i in np.arange(n_steps):
    (Xi,Si,Vi) = all_sol5[i]
    sol_to_plot[i] = Xi.dot(Si.dot(Vi.T))


anim2d = animation_2D(sol_to_plot, 'two_stream_5.gif', do_save=True)
HTML(anim2d.to_jshtml(fps=30))

## RANK 10 ANIMATION

In [ ]:
# ASSEMBLE SOLUTION
sol_to_plot = np.zeros((n_steps, nx, nv))
for i in np.arange(n_steps):
    (Xi,Si,Vi) = all_sol10[i]
    sol_to_plot[i] = Xi.dot(Si.dot(Vi.T))


anim2d = animation_2D(sol_to_plot, 'two_stream_10.gif', do_save=True)
HTML(anim2d.to_jshtml(fps=30))


## RANK 20 ANIMATION

In [ ]:
# ASSEMBLE SOLUTION
sol_to_plot = np.zeros((n_steps, nx, nv))
for i in np.arange(n_steps):
    (Xi,Si,Vi) = all_sol20[i]
    sol_to_plot[i] = Xi.dot(Si.dot(Vi.T))


anim2d = animation_2D(sol_to_plot, 'two_stream_20.gif', do_save=True)
HTML(anim2d.to_jshtml(fps=30))